In [ ]:
import pandas as pd
from cdc_ml.config import BOOKING_CYCLES_INTERIM,RECORDS_INTERIM
from cdc_ml.datasets.constants import TIMESLOTS
import holidays


In [ ]:
df_cycle = pd.read_parquet(BOOKING_CYCLES_INTERIM)

In [1061]:
df_cycle

,id,username,preference,range,cycle_end_reason,cycle_start,cycle_end
0,0,ajithak,"monday to saturday after 5pm , sunday anytime",till next week,completed,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00
1,1,addity,"weeday after 7 , no tues and thurs",before oct,completed,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00
2,2,addity,"anytime on weekends , weekday after 630pm",26 oct to 14 dec,completed,2025-10-17 00:00:00+08:00,2025-10-17 20:00:00+08:00
3,3,addity,"friday and weekends anytime, other weekdays af...",Jan and feb,completed,2026-01-16 20:00:00+08:00,2026-01-19 00:00:00+08:00
4,4,bryan,"before 25 sept , weekday after 730pm , 12 pm o...",NaN,completed,2025-08-22 00:48:00+08:00,2025-09-05 16:00:00+08:00
...,...,...,...,...,...,...,...
117,117,ali,whole apr anytime,NaN,NaN,2026-03-25 00:00:00+08:00,2026-04-10 23:59:00+08:00
118,118,ali,whole apr anytime,NaN,NaN,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00
119,119,bhara,"slot 5,6,7 all day 25 oct to 30 nov",NaN,NaN,2025-10-25 00:00:00+08:00,2025-10-30 23:59:00+08:00
120,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00


In [1062]:
df_cycle

,id,username,preference,range,cycle_end_reason,cycle_start,cycle_end
0,0,ajithak,"monday to saturday after 5pm , sunday anytime",till next week,completed,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00
1,1,addity,"weeday after 7 , no tues and thurs",before oct,completed,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00
2,2,addity,"anytime on weekends , weekday after 630pm",26 oct to 14 dec,completed,2025-10-17 00:00:00+08:00,2025-10-17 20:00:00+08:00
3,3,addity,"friday and weekends anytime, other weekdays af...",Jan and feb,completed,2026-01-16 20:00:00+08:00,2026-01-19 00:00:00+08:00
4,4,bryan,"before 25 sept , weekday after 730pm , 12 pm o...",NaN,completed,2025-08-22 00:48:00+08:00,2025-09-05 16:00:00+08:00
...,...,...,...,...,...,...,...
117,117,ali,whole apr anytime,NaN,NaN,2026-03-25 00:00:00+08:00,2026-04-10 23:59:00+08:00
118,118,ali,whole apr anytime,NaN,NaN,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00
119,119,bhara,"slot 5,6,7 all day 25 oct to 30 nov",NaN,NaN,2025-10-25 00:00:00+08:00,2025-10-30 23:59:00+08:00
120,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00


In [1063]:
def generate_rows (id:int,pref_s:str,pref_e:str,cycle_rules:dict[int,list[int]],interval="d", standard_timelist=TIMESLOTS):
    rows = []
    pref_start = pd.Timestamp(pref_s)
    pref_end = pd.Timestamp(pref_e)
    date_range = pd.date_range(pref_start,pref_end,freq=interval)
    
    timeslot_reference={}
    for i,time in enumerate(standard_timelist):
        timeslot_reference[i] = time

    for date in date_range:
        default_multi_hot = dict.fromkeys(TIMESLOTS,0)

        available_slots = cycle_rules[date.day_of_week]
        if len(available_slots)==0 :
            continue
        for slot in available_slots:
            time = timeslot_reference[slot]
            if time in default_multi_hot:
                default_multi_hot[time] = 1
        
        rows.append(
            {
            "id":id-1,
            "day_of_week":date.day_of_week,
            "day_name":date.day_name(),
            "pref_start":pref_start,
            "pref_end":pref_end,
            "date":date,


            **default_multi_hot
            }
            )
    return rows


In [ ]:
res = []
max_availability = (0,1,2,3,4,5,6)
DEFAULT_RULES:dict[int,tuple[int,...]] = {
    0:max_availability,
    1:max_availability,
    2:max_availability,
    3:max_availability,
    4:max_availability,
    5:max_availability,
    6:max_availability,
}

def get_rules(rules: dict[int,tuple[int, ...]], default_rules:dict[int,tuple[int,...]]=DEFAULT_RULES):
    result = {k: list(v) for k, v in default_rules.items()}
    if not rules:
        return result
    for day, timeslots in rules.items():
        result[day - 1] = [time - 1 for time in timeslots]
    return result







In [1065]:

pref_list = [

    (1,"2025-08-19","2025-08-31",get_rules(
       {
           1:(6,7),
           2:(6,7),
           3:(6,7),
           4:(6,7),
           5:(6,7),
           6:(6,7),
           
           } 
    )),
    (
        2,"2025-08-13","2025-09-30",get_rules(
            {
                1:(6,7,),
                2:(),
                3:(6,7,),
                4:(),
                5:(6,7,),
                6:(),
            }
        )
    ),
    (
        3,"2025-10-27","2025-12-14",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
            }

        )
    ),
    (
        4,"2026-01-16","2026-02-28",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
            }
        )
    ),
    (
        5,"2025-08-22","2025-09-24",get_rules(
            {
                1:(7,),
                2:(7,),
                3:(7,),
                4:(7,),
                5:(7,),
                6:(3,4,5,6,7),
                7:(3,4,5,6,7),

            }
        )
    ),

    (
        6,"2025-09-01","2025-09-02",get_rules(
            {
                1:(2,3,4,5),
                2:(2,3,4,5),
                3:(2,3,4,5),
                4:(2,3,4,5),
                5:(2,3,4,5),
                6:(2,3,4,5),
                7:(2,3,4,5),
            }
        )
    ),
    (
        7,"2025-08-18","2025-10-09",get_rules(
            {

                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(),
                6:(1,2,3),
                7:(1,2,3),

            }
        )
    ),

    (
        7,"2025-10-21","2025-10-31",get_rules(
            {

                1:(7,),
                2:(7,),
                3:(7,),
                4:(7,),
                5:(7,),
                6:(1,2,3),
                7:(1,2,3),

            }
        )
    ),
    (
        8,"2025-11-11","2025-12-31",get_rules(
            {
                1:(1,2),
                2:(1,2),
                3:(1,2),
                4:(1,2),
                5:(1,2),
            }
        )
    ),
    (
        9,"2025-09-04","2025-09-30",get_rules(
            {}
        )
    ),
    (
        10,"2025-09-02","2025-09-07",get_rules(
            {
                1:(),
                2:(),
                3:(),
                6:(),
                7:()
            }

        )
    ),
    (
        10,"2025-09-08","2025-09-14",get_rules(
            {}

        )
    ),
    (
        11,"2025-10-08","2025-11-30",get_rules({

            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(6,7),
        }
        )

    ),
    (
        12,"2025-10-20","2025-11-30",get_rules({

            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(6,7),
        }
        )

    ),
    (
        13,"2025-12-18","2026-01-31",get_rules({

            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
        }
        )

    ),

    (
        14,"2026-02-17","2026-02-28",get_rules({

            1:(),
            2:(),
            3:(),
            4:(),
            5:(),
            6:(4,5,6,7),
            7:(4,5,6,7),
        }
        )

    ),
    (
        15,"2025-09-14","2025-10-07",get_rules(
            {

            }
        )
    ),

    (
        15,"2025-10-19","2025-10-31",get_rules(
            {

            }
        )
    ),
    (
        16,"2025-11-01","2025-11-30",get_rules(
            {
                1:(7,),
                2:(7,),
                3:(7,),
                4:(7,),
                5:(7,),
                6:(5,6,7),
                7:(5,6,7),
            }
        )
    ),

    (
        17,"2025-11-01","2025-11-30",get_rules(
            {
                1:(7,),
                2:(7,),
                3:(7,),
                5:(7,),
                6:(5,6,7),
                7:(1,2,3,4,5,6,7),
            }
        )
    ),
    (
        18,"2025-12-01","2025-12-21",get_rules(
            {

                1:(7,),
                2:(7,),
                3:(7,),
                4:(7,),
                5:(7,),
                6:(),
            }
        )    
    ),
    (
        18,"2025-12-23","2026-01-04",get_rules(
            {

            }
        )    
    ),
    (
        19,"2025-12-08","2025-12-31",get_rules(
            {
                1:(7,),
                2:(7,),
                3:(7,),
                4:(7,),
                5:(7,),
                6:(),
            }
        )
    ),
    (
        20,"2026-01-25","2026-01-28",get_rules(
            {

            }
        )
    ),

    (
        21,"2026-02-14","2026-02-14",get_rules(
            {

            }
        )
    ),
    (
        21,"2026-02-16","2026-02-16",get_rules(
            {

            }
        )
    ),
    (
        21,"2026-02-20","2026-02-21",get_rules(
            {

            }
        )
    ),
    (
        21,"2026-02-23","2026-02-24",get_rules(
            {

            }
        )
    ),
    (
        22,"2025-09-08","2025-09-09",get_rules(
            {}
        )
    ),

    (
        22,"2025-09-12","2025-09-12",get_rules(
            {}
        )
    ),
    (
        22,"2025-09-15","2025-09-15",get_rules(
            {}
        )
    ),
    (
        22,"2025-09-11","2025-09-11",get_rules(
            {
                1:(1,2),
                2:(1,2),
                3:(1,2),
                4:(1,2),
                5:(1,2),
                6:(1,2),
                7:(1,2),
            }
        )
    ),
    (
        23,"2025-09-08","2025-09-11",get_rules(
            {
                1:(2,3,),
                2:(2,3,),
                3:(2,3,),
                4:(2,3,),
                5:(2,3,),
                6:(2,3,),
                7:(2,3,),
            }
        )
    ),
    (
        24,"2025-10-01","2025-10-01",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )
    ),
    (
        25,"2025-09-11","2025-10-31",get_rules(
            {
                1:(3,4,5,6,7),
                2:(3,4,5,6,7),
                3:(3,4,5,6,7),
                4:(3,4,5,6,7),
                5:(3,4,5,6,7),
                6:(3,4,5,6,7),
                7:(3,4,5,6,7),
            }
        ),"3d"
    ),
    (
        26,"2025-10-08","2025-10-31",get_rules(
            {
                1:(4,5,6,7),
                2:(4,5,6,7),
                3:(4,5,6,7),
                4:(4,5,6,7),
                5:(4,5,6,7),
                6:(4,5,6,7),
                7:(4,5,6,7),
            }
        ),"3d"
    ),
    (
        27,"2025-09-20","2025-09-20",get_rules(
            {
                1:(3,4,5,6,7),
                2:(3,4,5,6,7),
                3:(3,4,5,6,7),
                4:(3,4,5,6,7),
                5:(3,4,5,6,7),
                6:(3,4,5,6,7),
                7:(3,4,5,6,7),
            }
        )
    ),
    (
        27,"2025-09-21","2025-09-21",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )
    ),
    (
        27,"2025-09-26","2025-09-26",get_rules(
            {
                1:(5,6,7),
                2:(5,6,7),
                3:(5,6,7),
                4:(5,6,7),
                5:(5,6,7),
                6:(5,6,7),
                7:(5,6,7),
            }
        )
    ),
    (
       28,"2025-11-04","2025-11-30",get_rules(
           {
               1:(),
               2:(),
               3:(),
               4:(),
               5:(6,7),
           }
       ) 

    ),
    (
        29,"2025-11-18","2025-11-30",get_rules(

           {
               1:(),
               2:(),
               3:(),
               4:(),
               5:(6,7),
           }
        )
    ),
    (
        30,"2025-12-08","2025-11-30",get_rules(

           {
               1:(),
               2:(),
               3:(),
               4:(),
               5:(6,7),
           }
        )
    ),
    (
        31,"2025-09-22","2025-10-30",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
                6:(6,7),
                7:(6,7),

            }
        )
    ),
    (
        32,"2025-10-15","2025-11-30",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
                6:(6,7),
                7:(6,7),

            }
        )
    ),
    (
        33,"2025-10-20","2025-11-30",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
                6:(6,7),
                7:(6,7),

            }
        )
    ),
    (
        34,"2025-09-17","2025-09-30",get_rules(
            {
                1:(6,7,),
                2:(6,7,),
                3:(6,7,),
                4:(6,7,),
                5:(6,7,),
                6:(),
                7:(),

            }
        )
    ),
    (
        35,"2025-09-24","2025-10-24",get_rules(
            {
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
                6:(6,7),
                7:(6,7),
            }
        )
    ),
    (
        36,"2025-10-05","2025-11-30",get_rules(
            {

            }
        )
    ),
    (
        37,"2025-11-14","2025-12-31",get_rules(
            {

            }
        )
    ),
    (
        38,"2025-11-26","2025-12-31",get_rules({

        })
    ),
    (
        39,"2025-12-24","2026-01-30",get_rules(
            {}
        )
    ),

    (
        40,"2026-01-07","2026-01-30",get_rules(
            {}
        )
    ),
    (
        41,"2026-01-30","2026-02-28",get_rules(
            {}
        )
    ),
    (
        42,"2025-10-03","2025-10-03",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        42,"2025-10-10","2025-10-10",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        42,"2025-10-17","2025-10-17",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        42,"2025-10-31","2025-10-31",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        42,"2025-11-07","2025-11-07",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        42,"2025-11-14","2025-11-14",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),


    (
        42,"2025-11-28","2025-11-28",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        43,"2025-10-10","2025-10-10",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        43,"2025-10-31","2025-10-31",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        43,"2025-11-07","2025-11-07",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        43,"2025-11-14","2025-11-14",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),


    (
        43,"2025-11-28","2025-11-28",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        44,"2025-11-07","2025-11-07",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )

    ),

    (
        44,"2025-11-14","2025-11-14",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )

    ),


    (
        44,"2025-11-28","2025-11-28",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2025-12-05","2025-12-05",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        45,"2025-12-12","2025-12-12",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        45,"2025-12-19","2025-12-19",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2025-12-26","2025-12-26",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2025-12-29","2025-12-30",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2026-01-02","2026-01-02",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        45,"2026-01-09","2026-01-09",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2026-01-16","2026-01-16",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2026-01-23","2026-01-23",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        45,"2026-01-29","2026-01-29",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        46,"2025-12-29","2025-12-30",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        46,"2026-01-02","2026-01-02",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),
    (
        46,"2026-01-09","2026-01-09",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )

    ),

    (
        46,"2026-01-11","2026-01-11",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )

    ),
    (
        46,"2026-01-16","2026-01-16",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        46,"2026-01-23","2026-01-23",get_rules(
            {
                1:(1,2,3,4),
                2:(1,2,3,4),
                3:(1,2,3,4),
                4:(1,2,3,4),
                5:(1,2,3,4),
                6:(1,2,3,4),
                7:(1,2,3,4),
            }
        )

    ),

    (
        46,"2026-01-30","2026-01-30",get_rules(
            {
                1:(1,2,3,4,5),
                2:(1,2,3,4,5),
                3:(1,2,3,4,5),
                4:(1,2,3,4,5),
                5:(1,2,3,4,5),
                6:(1,2,3,4,5),
                7:(1,2,3,4,5),
            }
        )

    ),
    
    (
        47,"2026-01-09","2026-01-09",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),


    (
        47,"2026-01-16","2026-01-16",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),

    (
        47,"2026-01-23","2026-01-23",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),

    (
        47,"2026-01-30","2026-01-30",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-03","2026-02-03",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-06","2026-02-06",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-10","2026-02-10",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-13","2026-02-13",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-20","2026-02-20",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        47,"2026-02-27","2026-02-27",get_rules(
            {
                1:(3,4),
                2:(3,4),
                3:(3,4),
                4:(3,4),
                5:(3,4),
                6:(3,4),
                7:(3,4),
            }
        )

    ),
    (
        48,"2026-02-06","2026-02-06",get_rules(
            {
                1:(3,),
                2:(3,),
                3:(3,),
                4:(3,),
                5:(3,),
                6:(3,),
                7:(3,),
            }
        )

    ),
    (
        48,"2026-02-20","2026-02-20",get_rules(
            {
                1:(3,),
                2:(3,),
                3:(3,),
                4:(3,),
                5:(3,),
                6:(3,),
                7:(3,),
            }
        )

    ),
    (
        48,"2026-03-03","2026-03-03",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        48,"2026-03-10","2026-03-10",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        48,"2026-03-13","2026-03-13",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        48,"2026-03-24","2026-03-24",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        48,"2026-03-27","2026-03-27",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        48,"2026-03-31","2026-03-31",get_rules(
            {
                1:(),
                2:(6,),
                3:(),
                4:(),
                5:(3,),
                6:(),
                7:(),
            }
        )

    ),
    (
        49,"2026-04-01","2026-04-01",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        49,"2026-04-07","2026-04-07",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        49,"2026-04-14","2026-04-14",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        49,"2026-04-21","2026-04-21",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        49,"2026-04-28","2026-04-29",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        49,"2026-04-03","2026-04-03",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        49,"2026-04-10","2026-04-10",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        49,"2026-04-17","2026-04-17",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        49,"2026-04-24","2026-04-24",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),


    (
        50,"2026-04-01","2026-04-01",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        50,"2026-04-07","2026-04-07",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        50,"2026-04-14","2026-04-14",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        50,"2026-04-21","2026-04-21",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        50,"2026-04-28","2026-04-29",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),

        })
    ),
    (
        50,"2026-04-03","2026-04-03",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        50,"2026-04-10","2026-04-10",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        50,"2026-04-17","2026-04-17",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        50,"2026-04-24","2026-04-24",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),

    (
        51,"2025-10-17","2025-10-25",get_rules({
            1:(3,),
            2:(3,),
            3:(3,),
            4:(3,),
            5:(3,),
            6:(3,),
            7:(3,),
            

        })
    ),
    (
        51,"2025-10-21","2025-10-25",get_rules({
            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(7,),
            6:(7,),
            7:(7,),

        })
    ),

    (
        52,"2025-12-01","2025-12-03",get_rules({
            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(7,),
            6:(7,),
            7:(7,),

        })
    ),
    (
        53,"2025-10-31","2025-11-03",get_rules({
            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        53,"2025-11-05","2025-11-08",get_rules({
            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        53,"2025-11-10","2025-11-12",get_rules({
            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),


    (
        53,"2025-11-14","2025-11-21",get_rules({
            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        54,"2025-12-05","2025-12-26",get_rules({

            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        55,"2025-12-05","2025-12-26",get_rules({

            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),


    (
        56,"2025-12-05","2025-12-26",get_rules({

            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        57,"2026-01-01","2026-01-30",get_rules({

            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        58,"2026-02-01","2026-02-28",get_rules({

            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
        })
    ),
    (
        59,"2026-03-09","2026-03-31",get_rules({

            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(),
            7:(),

        })
    ),
    (

        60,"2026-04-02","2026-04-02",get_rules({
            1:(1,2,3,4),
            2:(1,2,3,4),
            3:(1,2,3,4),
            4:(1,2,3,4),
            5:(1,2,3,4),
            6:(1,2,3,4),
            7:(1,2,3,4),
            

        })
    ),


    (

        60,"2026-04-06","2026-04-06",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(1,3,4),
            7:(1,3,4),
            

        })
    ),
    (

        60,"2026-04-13","2026-04-13",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(1,3,4),
            7:(1,3,4),
            

        })
    ),

    (

        60,"2026-04-02","2026-04-02",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(1,3,4),
            7:(1,3,4),
            

        })
    ),


    (

        60,"2026-04-09","2026-04-09",get_rules({
            1:(1,),
            2:(1,),
            3:(1,),
            4:(1,),
            5:(1,),
            6:(1,),
            7:(1,),
            

        })
    ),


    (

        60,"2026-04-10","2026-04-10",get_rules({
            1:(1,),
            2:(1,),
            3:(1,),
            4:(1,),
            5:(1,),
            6:(1,),
            7:(1,),
            

        })
    ),

    (

        60,"2026-04-11","2026-04-11",get_rules({
            1:(1,),
            2:(1,),
            3:(1,),
            4:(1,),
            5:(1,),
            6:(1,),
            7:(1,),
            

        })
    ),
    (

        60,"2026-04-15","2026-04-15",get_rules({
            1:(1,),
            2:(1,),
            3:(1,),
            4:(1,),
            5:(1,),
            6:(1,),
            7:(1,),
            

        })
    ),
    (
        61,"2025-11-03","2025-11-30",get_rules({})
    ),
    (
        62,"2025-11-07","2025-11-30",get_rules({})
    ),
    (
        63,"2026-01-01","2026-01-30",get_rules({})
    ),
    (
        64,"2026-01-21","2026-02-21",get_rules({})
    ),
    (
        65,"2025-11-10","2025-11-11",get_rules({

        })
    ),

    (
        65,"2025-11-15","2025-11-20",get_rules({

        2:(4,5,6,7),
        })
    ),
    (
        65,"2025-11-22","2025-11-30",get_rules({

        2:(4,5,6,7),
        6:(),
        7:(3,4,5,6,7)
        })
    ),
    (
        65,"2025-12-01","2025-12-07",get_rules({
            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2),
            7:(),
        })
    ),

    (
        65,"2025-12-09","2025-12-10",get_rules({
            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2),
            7:(),
        })
    ),
    (
        65,"2025-12-15","2025-12-31",get_rules({
            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2),
            7:(),
        })
    ),
    (
        65,"2025-12-13","2025-12-31",get_rules({
            1:(1,2,3,4,5,6),
            2:(1,2,3,4,5,6),
            3:(1,2,3,4,5,6),
            4:(1,2,3,4,5,6),
            5:(1,2,3,4,5,6),
            6:(1,2,),
            7:(1,2,),
        })
    ),
    (

        66,"2026-01-07","2026-01-07",get_rules({


        })
    ),
    (

        66,"2026-01-09","2026-01-09",get_rules({


        })
    ),
    (

        66,"2026-01-12","2026-01-31",get_rules({
            1:(6,),
            2:(),
            3:(),
            4:(),
            5:(6,7),
            6:(),
            7:(),

        })
    ),
    (

        66,"2026-01-17","2026-01-31",get_rules({
            1:(6,7),
            2:(),
            3:(),
            4:(),
            5:(),
            6:(1,2,6,7),
            7:(),
        })
    ),

    (

        66,"2026-02-01","2026-02-28",get_rules({
            1:(6,),
            2:(),
            3:(),
            4:(),
            5:(6,7),
            6:(1,2),
            7:(),
        })
    ),
    (
        67,"2025-11-17","2025-11-21",get_rules({

            1:(2,4,6,7),
            2:(2,4,6,7),
            3:(2,4,6,7),
            4:(2,4,6,7),
            5:(2,4,6,7),
            6:(2,4,6,7),
            7:(2,4,6,7),

        })
    ),
    (
        67,"2025-11-27","2025-11-28",get_rules({

            1:(2,4),
            2:(2,4),
            3:(2,4),
            4:(2,4),
            5:(2,4),
            6:(2,4),
            7:(2,4),

        })
    ),

    (
        68,"2025-11-17","2025-11-21",get_rules({


        })
    ),
    (
        68,"2025-11-21","2025-11-23",get_rules({


        })
    ),
    (
        68,"2025-11-27","2025-11-28",get_rules({


        })
    ),
    (
        69,"2025-11-22","2025-11-23",get_rules({

        })
    ),
    (
        69,"2025-11-27","2025-11-27",get_rules({

        })
    ),
    (
        70,"2026-01-06","2026-01-07",get_rules({


        })
    ),
    (
        71,"2025-11-07","2025-12-31",get_rules({


        })
    ),
    (
        72,"2025-11-23","2025-12-31",get_rules({


        })
    ),
    (
        73,"2025-11-25","2025-12-31",get_rules({


        })
    ),
    (
        74,"2025-11-22","2025-11-22",get_rules({
            1:(3,),
            2:(3,),
            3:(3,),
            4:(3,),
            5:(3,),
            6:(3,),
            7:(3,),
        })

    ),
    (
        74,"2025-11-18","2025-11-18",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),
            
        })

    ),
    (
        75,"2025-11-21","2025-11-21",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),
        })
    ),
    (
        75,"2025-11-22","2025-11-22",get_rules({
            1:(3,4),
            2:(3,4),
            3:(3,4),
            4:(3,4),
            5:(3,4),
            6:(3,4),
            7:(3,4),
        })
    ),
    (
        75,"2025-11-25","2025-11-25",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),
        })
    ),
    (
        75,"2025-11-28","2025-11-28",get_rules({
            1:(5,6,),
            2:(5,6,),
            3:(5,6,),
            4:(5,6,),
            5:(5,6,),
            6:(5,6,),
            7:(5,6,),
            7:(5,6,),
        })
    ),
    (
        75,"2025-11-29","2025-11-29",get_rules({
            1:(3,4),
            2:(3,4),
            3:(3,4),
            4:(3,4),
            5:(3,4),
            6:(3,4),
            7:(3,4),
        })
    ),
    (
        76,"2025-12-01","2025-12-05",get_rules({
            1:(5,6),
            2:(5,6),
            3:(5,6),
            4:(5,6),
            5:(5,6),
            6:(5,6),
            7:(5,6),
        })
    ),
    (
        77,"2026-01-01","2026-01-02",get_rules({
            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),
        })
    ),
    (
        78,"2026-01-19","2026-01-22",get_rules({

            1:(6,),
            2:(6,),
            3:(6,),
            4:(6,),
            5:(6,),
            6:(6,),
            7:(6,),

        })
    ),
    (
        79,"2026-02-09","2026-02-13",get_rules({
            1:(2,),
            2:(2,),
            3:(2,),
            4:(2,),
            5:(2,),
            6:(2,),
            7:(2,),
        })
    ),
    (
        80,"2026-02-23","2026-02-27",get_rules({
            1:(1,3),
            2:(1,3),
            3:(1,3),
            4:(1,3),
            5:(1,3),
            6:(1,3),
            7:(1,3),
        })
    ),
    (
        81,"2026-03-23","2026-03-27",get_rules({
            1:(3,4,5),
            2:(3,4,5),
            3:(3,4,5),
            4:(3,4,5),
            5:(3,4,5),
            6:(3,4,5),
            7:(3,4,5),

        })
    ),
    (
        82,"2026-03-23","2026-03-27",get_rules({
            1:(3,4,5),
            2:(3,4,5),
            3:(3,4,5),
            4:(3,4,5),
            5:(3,4,5),
            6:(3,4,5),
            7:(3,4,5),

        })
    ),
    (
        83,"2026-03-30","2026-04-01",get_rules({
            1:(5,),
            2:(5,),
            3:(5,),
            4:(5,),
            5:(5,),
            6:(5,),
            7:(5,),
        })
    ),
    (
        84,"2026-04-06","2026-04-10",get_rules({

            1:(5,),
            2:(5,),
            3:(5,),
            4:(5,),
            5:(5,),
            6:(5,),
            7:(5,),
        })
    ),
    (
        85,"2025-12-01","2025-12-11",get_rules({
            1:(1,2,3,4,5,6,7),
            2:(1,2,3,4,5,6,7),
            3:(1,2,3,4,5,6,7),
            4:(1,2,3,4,5,6,7),
            5:(1,2,3,4,5,6,7),
            6:(),
            7:(),

        })
    ),
    (
        85,"2025-12-22","2025-12-23",get_rules({
            1:(1,2,3,4,5,6,7),
            2:(1,2,3,4,5,6,7),
            3:(1,2,3,4,5,6,7),
            4:(1,2,3,4,5,6,7),
            5:(1,2,3,4,5,6,7),
            6:(),
            7:(),

        })
    ),
    (
        85,"2025-12-26","2025-12-26",get_rules({
            1:(1,2,3,4,5,6,7),
            2:(1,2,3,4,5,6,7),
            3:(1,2,3,4,5,6,7),
            4:(1,2,3,4,5,6,7),
            5:(1,2,3,4,5,6,7),
            6:(),
            7:(),

        })
    ),
    (
        85,"2025-12-29","2025-12-30",get_rules({
            1:(1,2,3,4,5,6,7),
            2:(1,2,3,4,5,6,7),
            3:(1,2,3,4,5,6,7),
            4:(1,2,3,4,5,6,7),
            5:(1,2,3,4,5,6,7),
            6:(),
            7:(),

        })
    ),
    (
        86,"2026-01-02","2026-01-02",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        86,"2026-01-08","2026-01-09",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        86,"2026-01-15","2026-01-16",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        86,"2026-01-22","2026-01-22",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        86,"2026-01-29","2026-01-30",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        87,"2026-01-15","2026-01-16",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        87,"2026-01-22","2026-01-22",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        88,"2026-02-02","2026-02-02",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),

        })
    ),
    (
        88,"2026-02-05","2026-02-05",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),

        })
    ),
    (
        88,"2026-02-09","2026-02-09",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),

        })
    ),
    (
        88,"2026-02-10","2026-02-10",get_rules({

        })
    ),
    (
        88,"2026-02-12","2026-02-12",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),
            

        })
    ),

    (
        88,"2026-02-24","2026-02-24",get_rules({
            1:(3,4,5),
            2:(3,4,5),
            3:(3,4,5),
            4:(3,4,5),
            5:(3,4,5),
            6:(3,4,5),
            7:(3,4,5),
            

        })
    ),
    (
        88,"2026-02-27","2026-02-27",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),

        })
    ),
    (
        89,"2026-04-01","2026-04-15",get_rules({
            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),

        })
    ),
    (
        90,"2025-12-04","2026-12-05",get_rules({

        })
    ),

    (
        90,"2025-12-08","2026-12-08",get_rules({

        })
    ),
    (
        91,"2025-12-04","2026-12-05",get_rules({

        })
    ),
    (
        91,"2025-12-08","2026-12-08",get_rules({

        })
    ),
    (
        92,"2026-01-01","2026-01-30",get_rules({

        })
    ),
    (
        93,"2026-02-17","2026-02-28",get_rules({
            1:(2,3,4,5,6,7),
            2:(2,3,4,5,6,7),
            3:(2,3,4,5,6,7),
            4:(2,3,4,5,6,7),
            5:(2,3,4,5,6,7),
            6:(2,3,4,5,6,7),
            7:(2,3,4,5,6,7),

        })
    ),

    (
        94,"2026-02-17","2026-02-28",get_rules({
            1:(2,3,4,5,6,7),
            2:(2,3,4,5,6,7),
            3:(2,3,4,5,6,7),
            4:(2,3,4,5,6,7),
            5:(2,3,4,5,6,7),
            6:(2,3,4,5,6,7),
            7:(2,3,4,5,6,7),

        })
    ),

    (
        95,"2026-03-02","2026-03-31",get_rules({
            1:(2,3,4,5,6,7),
            2:(2,3,4,5,6,7),
            3:(2,3,4,5,6,7),
            4:(2,3,4,5,6,7),
            5:(2,3,4,5,6,7),
            6:(2,3,4,5,6,7),
            7:(2,3,4,5,6,7),

        })
    ),
    (
        96,"2026-04-01","2026-04-30",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            6:(6,7),

        })
    ),
    (
        97,"2025-12-18","2025-12-31",get_rules({
        })
    ),
    (
        97,"2026-01-01","2026-01-30",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(1,2),
            7:(1,2),
        })
    ),

    (
        98,"2026-01-10","2026-01-30",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(1,2),
            7:(1,2),
        })
    ),
    (
        99,"2025-12-30","2026-01-11",get_rules({
            1:(2,3,4,5,6),
            2:(2,3,4,5,6),
            3:(2,3,4,5,6),
            4:(2,3,4,5,6),
            5:(2,3,4,5,6),
            6:(2,3,4,5,6),
            7:(2,3,4,5,6),
        })
    ),
    (
        99,"2026-02-03","2026-02-03",get_rules({
            1:(1,),
            2:(1,),
            3:(1,),
            4:(1,),
            5:(1,),
            6:(1,),
            7:(1,),
            
        })
    ),
    (
        100,"2026-01-17","2026-01-18",get_rules({

            1:(2,3,4,5,6),
            2:(2,3,4,5,6),
            3:(2,3,4,5,6),
            4:(2,3,4,5,6),
            5:(2,3,4,5,6),
            6:(2,3,4,5,6),
            7:(2,3,4,5,6),
        })
    ),

    (
        101,"2025-11-27","2025-12-30",get_rules({
            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(7,),
            6:(7,),
            7:(7,),
        })
    ),
    (
        102,"2025-12-07","2025-12-30",get_rules({
            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(7,),
            6:(7,),
            7:(7,),
        })
    ),
    (
        103,"2026-01-06","2026-02-28",get_rules({
            1:(6,7,),
            2:(6,7,),
            3:(6,7,),
            4:(6,7,),
            5:(6,7,),
            6:(6,7,),
            7:(6,7,),
        })
    ),
    (
        104,"2026-01-23","2026-01-23",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    ( 104,"2026-01-26","2026-01-26",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    ( 104,"2026-01-30","2026-01-30",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        105,"2026-02-10","2026-02-13",get_rules({
            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),
        })
    ),
    (
        106,"2026-02-16","2026-02-16",get_rules({

            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),
        })
    ),

    (
        106,"2026-02-24","2026-02-24",get_rules({

            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),
        })
    ),
    (
        106,"2026-02-11","2026-02-12",get_rules({

        })
    ),
    (
        106,"2026-02-18","2026-02-18",get_rules({

        })
    ),
    (
        106,"2026-02-23","2026-02-23",get_rules({

        })
    ),

    (
        106,"2026-02-19","2026-02-20",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        106,"2026-02-25","2026-02-25",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        106,"2026-02-26","2026-02-26",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        106,"2026-02-09","2026-02-09",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),
            

        })
    ),
    (
        106,"2026-02-27","2026-02-27",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),
            

        })
    ),




    (
        107,"2026-02-16","2026-02-16",get_rules({

            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),
        })
    ),

    (
        107,"2026-02-24","2026-02-24",get_rules({

            1:(1,2,3,4,5),
            2:(1,2,3,4,5),
            3:(1,2,3,4,5),
            4:(1,2,3,4,5),
            5:(1,2,3,4,5),
            6:(1,2,3,4,5),
            7:(1,2,3,4,5),
        })
    ),
    (
        107,"2026-02-18","2026-02-18",get_rules({

        })
    ),
    (
        107,"2026-02-23","2026-02-23",get_rules({

        })
    ),

    (
        107,"2026-02-19","2026-02-20",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        107,"2026-02-25","2026-02-25",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        107,"2026-02-26","2026-02-26",get_rules({
            1:(1,2),
            2:(1,2),
            3:(1,2),
            4:(1,2),
            5:(1,2),
            6:(1,2),
            7:(1,2),

        })
    ),
    (
        107,"2026-02-27","2026-02-27",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),
            

        })
    ),

    (
        108,"2026-03-14","2026-03-14",get_rules({
            1:(4,5,6,7),
            2:(4,5,6,7),
            3:(4,5,6,7),
            4:(4,5,6,7),
            5:(4,5,6,7),
            6:(4,5,6,7),
            7:(4,5,6,7),
        })
    ),

    (
        108,"2026-03-16","2026-03-28",get_rules({
            1:(),
            2:(),
            3:(),
            4:(),
            5:(),
        })
    ),

    (
        109,"2026-03-14","2026-03-14",get_rules({
            1:(4,5,6,7),
            2:(4,5,6,7),
            3:(4,5,6,7),
            4:(4,5,6,7),
            5:(4,5,6,7),
            6:(4,5,6,7),
            7:(4,5,6,7),
        })
    ),

    (
        109,"2026-03-16","2026-03-28",get_rules({
            1:(),
            2:(),
            3:(),
            4:(),
            5:(),
        })
    ),

    (
       110,"2026-04-18","2026-04-19",get_rules({


       }) 
    ),

    (
       110,"2026-04-26","2026-04-26",get_rules({


       }) 
    ),

    (
        111,"2026-03-21","2026-05-31",get_rules({
            1:(7,),
            2:(7,),
            3:(7,),
            4:(7,),
            5:(7,),
        })
    ),

    (
        112,"2025-11-01","2025-11-30",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(),
            7:(),

        })
    ),
    (
        113,"2025-12-01","2025-12-31",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(),
            7:(),

        })
    ),
    (
        114,"2025-12-17","2025-12-31",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(),
            7:(),

        })
    ),
    (
        115,"2026-01-08","2026-01-30",get_rules({
            1:(1,3,4),
            2:(1,3,4),
            3:(1,3,4),
            4:(1,3,4),
            5:(1,3,4),
            6:(),
            7:(),

        })
    ),
    (
        116,"2026-02-06","2026-02-28",get_rules({
        })
    ),
    (
        117,"2026-03-02","2026-03-31",get_rules({
        })
    ),
    (
        118,"2026-04-01","2026-04-30",get_rules({
        })
    ),
    (
        119,"2026-04-21","2026-04-30",get_rules({
        })
    ),
    (
        120,"2025-10-25","2025-11-30",get_rules({
            1:(5,6,7),
            2:(5,6,7),
            3:(5,6,7),
            4:(5,6,7),
            5:(5,6,7),
            6:(5,6,7),
            7:(5,6,7),
        })
    ),
    (
        121,"2025-10-27","2025-12-31",get_rules({
            1:(6,7),
            2:(6,7),
            3:(6,7),
            4:(6,7),
            5:(6,7),
            6:(6,7),
            7:(6,7),

        })
    ),
    (
        122,"2025-10-16","2025-12-14",get_rules(
            {
                1:(6,7),
                2:(6,7),
                3:(6,7),
                4:(6,7),
                5:(6,7),
            }

        )
    ),

]
# 1.0830
# 2.1020
# 3.1245
# 4.1435
# 5.1625
# 6.1850
# 7.2040


In [1066]:

for pref in pref_list:
    if len(pref)==5:
        res.extend(generate_rows(pref[0],pref[1],pref[2],pref[3],pref[4]))
    else :
        res.extend(generate_rows(pref[0],pref[1],pref[2],pref[3]))

C:\Users\zhiju\AppData\Local\Temp\ipykernel_1156\1404918059.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  date_range = pd.date_range(pref_start,pref_end,freq=interval)


In [1067]:
df = pd.DataFrame(res)

In [1068]:
df.loc[df["id"]==26]

,id,day_of_week,day_name,pref_start,pref_end,date,08:30,10:20,12:45,14:35,16:25,18:50,20:40
652,26,5,Saturday,2025-09-20,2025-09-20,2025-09-20,0,0,1,1,1,1,1
653,26,6,Sunday,2025-09-21,2025-09-21,2025-09-21,1,1,1,1,1,0,0
654,26,4,Friday,2025-09-26,2025-09-26,2025-09-26,0,0,0,0,1,1,1


In [1069]:
df.tail()

,id,day_of_week,day_name,pref_start,pref_end,date,08:30,10:20,12:45,14:35,16:25,18:50,20:40
3980,121,2,Wednesday,2025-10-16,2025-12-14,2025-12-10,0,0,0,0,0,1,1
3981,121,3,Thursday,2025-10-16,2025-12-14,2025-12-11,0,0,0,0,0,1,1
3982,121,4,Friday,2025-10-16,2025-12-14,2025-12-12,0,0,0,0,0,1,1
3983,121,5,Saturday,2025-10-16,2025-12-14,2025-12-13,1,1,1,1,1,1,1
3984,121,6,Sunday,2025-10-16,2025-12-14,2025-12-14,1,1,1,1,1,1,1


In [1070]:
df = df.rename(columns={"08:30":"t_0830","10:20":"t_1020","12:45":"t_1245","14:35":"t_1435","16:25":"t_1625","18:50":"t_1850","20:40":"t_2040"})

In [1071]:
df_merge = df_cycle.merge(df,on="id",how="left")

In [1072]:
df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 3986 entries, 0 to 3985
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype                         
---  ------            --------------  -----                         
 0   id                3986 non-null   int64                         
 1   username          3986 non-null   str                           
 2   preference        3597 non-null   str                           
 3   range             2891 non-null   str                           
 4   cycle_end_reason  3611 non-null   str                           
 5   cycle_start       3986 non-null   datetime64[ns, Asia/Singapore]
 6   cycle_end         3986 non-null   datetime64[ns, Asia/Singapore]
 7   day_of_week       3985 non-null   float64                       
 8   day_name          3985 non-null   str                           
 9   pref_start        3985 non-null   datetime64[us]                
 10  pref_end          3985 non-null   datetime64[us]           

In [1073]:
invalid = df_merge.loc[pd.to_datetime(df_merge["pref_start"]).dt.tz_localize("Asia/Singapore").dt.floor("D") <df_merge["cycle_start"].dt.floor("D"),["id","username","cycle_start","pref_start"]]
print(invalid["id"].unique())

[90]


In [1074]:
df_merge.loc[df_merge["id"]==120]

,id,username,preference,range,cycle_end_reason,cycle_start,cycle_end,day_of_week,day_name,pref_start,pref_end,date,t_0830,t_1020,t_1245,t_1435,t_1625,t_1850,t_2040
3860,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,0.0,Monday,2025-10-27,2025-12-31,2025-10-27,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3861,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,1.0,Tuesday,2025-10-27,2025-12-31,2025-10-28,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3862,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,2.0,Wednesday,2025-10-27,2025-12-31,2025-10-29,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3863,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,3.0,Thursday,2025-10-27,2025-12-31,2025-10-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3864,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,4.0,Friday,2025-10-27,2025-12-31,2025-10-31,0.0,0.0,0.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3921,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,5.0,Saturday,2025-10-27,2025-12-31,2025-12-27,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3922,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,6.0,Sunday,2025-10-27,2025-12-31,2025-12-28,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3923,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,0.0,Monday,2025-10-27,2025-12-31,2025-12-29,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3924,120,d,"27 oct to 31 dec , 6,7",NaN,NaN,2025-10-25 00:00:00+08:00,2025-11-04 23:59:00+08:00,1.0,Tuesday,2025-10-27,2025-12-31,2025-12-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [1075]:
df.loc[df["pref_end"]<df["pref_start"]]

,id,day_of_week,day_name,pref_start,pref_end,date,t_0830,t_1020,t_1245,t_1435,t_1625,t_1850,t_2040


In [ ]:
df_records = pd.read_parquet(RECORDS_INTERIM)

In [1077]:
df_records.loc[df_records["username"]=="addity"]

,id,username,lesson_at,booking_at,booking_type
2,2,addity,2025-09-21 14:35:00+08:00,2025-08-23 00:57:26.576198+08:00,0
9,9,addity,2025-09-03 18:50:00+08:00,2025-08-28 06:13:21.405627+08:00,0
51,51,addity,2025-10-18 08:30:00+08:00,2025-10-16 23:18:27.344009+08:00,0
54,54,addity,2025-12-01 18:50:00+08:00,2025-10-17 18:52:45.642725+08:00,0
55,55,addity,2025-12-02 18:50:00+08:00,2025-10-17 18:53:10.999728+08:00,0
56,56,addity,2025-12-01 20:40:00+08:00,2025-10-17 18:54:07.486667+08:00,0
57,57,addity,2025-12-02 20:40:00+08:00,2025-10-17 18:54:37.367113+08:00,0
58,58,addity,2025-12-06 12:45:00+08:00,2025-10-17 18:55:31.170682+08:00,0
59,59,addity,2025-12-06 12:45:00+08:00,2025-10-17 18:55:31.247233+08:00,0


In [1078]:
df_records

,id,username,lesson_at,booking_at,booking_type
0,0,ajithak,2025-08-28 18:50:00+08:00,2025-08-21 09:24:06.300383+08:00,0
1,1,ajithak,2025-08-28 20:40:00+08:00,2025-08-21 09:24:16.048708+08:00,0
2,2,addity,2025-09-21 14:35:00+08:00,2025-08-23 00:57:26.576198+08:00,0
3,3,bryan,2025-08-28 20:40:00+08:00,2025-08-25 12:06:06.539664+08:00,0
4,4,bryan,2025-08-31 12:45:00+08:00,2025-08-26 12:50:24.317931+08:00,0
...,...,...,...,...,...
519,519,tomato,NaT,2026-01-01 00:01:00+08:00,1
520,520,bhara,2025-11-04 18:50:00+08:00,2025-10-28 13:53:00+08:00,1
521,521,d,2025-11-07 18:50:00+08:00,2025-10-28 17:10:00+08:00,1
522,522,bhara,2025-11-03 16:25:00+08:00,2025-10-29 15:56:00+08:00,1


In [1079]:
rows = []
for row in df_merge.itertuples():
    times =[]
    if row.t_0830 == 1.0:
        times.append("08:30")
    if row.t_1020 == 1.0:
        times.append("10:20")
    if row.t_1245 == 1.0:
        times.append("12:45")
    if row.t_1435 == 1.0:
        times.append("14:35")
    if row.t_1625 == 1.0:
        times.append("16:25")
    if row.t_1850 == 1.0:
        times.append("18:50")
    if row.t_2040 == 1.0:
        times.append("20:40")
   
    for time in times:
        rows.append({
            "id":row.id,
            "username":row.username,
            "pref_at":pd.to_datetime(str(row.date) + " " + time).tz_localize("Asia/Singapore")
        })

df_2 = pd.DataFrame(rows)


In [1080]:
df_missing = df_records.merge(df_2,left_on=["username","lesson_at"],right_on=["username","pref_at"],how="left")

In [1081]:
df_missing

,id_x,username,lesson_at,booking_at,booking_type,id_y,pref_at
0,0,ajithak,2025-08-28 18:50:00+08:00,2025-08-21 09:24:06.300383+08:00,0,0.0,2025-08-28 18:50:00+08:00
1,1,ajithak,2025-08-28 20:40:00+08:00,2025-08-21 09:24:16.048708+08:00,0,0.0,2025-08-28 20:40:00+08:00
2,2,addity,2025-09-21 14:35:00+08:00,2025-08-23 00:57:26.576198+08:00,0,1.0,2025-09-21 14:35:00+08:00
3,3,bryan,2025-08-28 20:40:00+08:00,2025-08-25 12:06:06.539664+08:00,0,4.0,2025-08-28 20:40:00+08:00
4,4,bryan,2025-08-31 12:45:00+08:00,2025-08-26 12:50:24.317931+08:00,0,4.0,2025-08-31 12:45:00+08:00
...,...,...,...,...,...,...,...
735,519,tomato,NaT,2026-01-01 00:01:00+08:00,1,NaN,NaT
736,520,bhara,2025-11-04 18:50:00+08:00,2025-10-28 13:53:00+08:00,1,119.0,2025-11-04 18:50:00+08:00
737,521,d,2025-11-07 18:50:00+08:00,2025-10-28 17:10:00+08:00,1,120.0,2025-11-07 18:50:00+08:00
738,522,bhara,2025-11-03 16:25:00+08:00,2025-10-29 15:56:00+08:00,1,119.0,2025-11-03 16:25:00+08:00


In [1082]:
df_merge.loc[(df_merge["username"]=="addity") & df_merge["date"].between("2025-09-01","2025-09-30")
             ,["username","date","pref_start","pref_end","t_0830",	"t_1020",	"t_1245",	"t_1435",	"t_1625",	"t_1850",	"t_2040"]]

,username,date,pref_start,pref_end,t_0830,t_1020,t_1245,t_1435,t_1625,t_1850,t_2040
24,addity,2025-09-01,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
25,addity,2025-09-03,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
26,addity,2025-09-05,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
27,addity,2025-09-07,2025-08-13,2025-09-30,1.0,1.0,1.0,1.0,1.0,1.0,1.0
28,addity,2025-09-08,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
29,addity,2025-09-10,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
30,addity,2025-09-12,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
31,addity,2025-09-14,2025-08-13,2025-09-30,1.0,1.0,1.0,1.0,1.0,1.0,1.0
32,addity,2025-09-15,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0
33,addity,2025-09-17,2025-08-13,2025-09-30,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [1083]:
df_missing.loc[ (df_missing["pref_at"].isna()) & ~df_missing["lesson_at"].isna(),["username","lesson_at","booking_at"]].sort_values(by="username")

,username,lesson_at,booking_at
